In [1]:
!pip3 install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
df = pd.read_csv("../DataSets/MovieReviews_Lemmatized.csv")

def join_tokens(text):
    if pd.isna(text):
        return ""
    try:
        tokens = ast.literal_eval(text)
        return " ".join(tokens)
    except:
        return text

df["Cleaned_Text"] = df["Reviews"].apply(join_tokens)

df = df.dropna(subset=['Cleaned_Text', 'emotion'])

print("Veri boyutu:", df.shape)
df[['Cleaned_Text', 'emotion']].head(3)

Veri boyutu: (19316, 5)


,Cleaned_Text,emotion
0,laugh overall motivation character incomprehen...,anticipation
1,wait exhale wait wait wait wait get point wait...,anticipation
2,angela basset good expect whitney range actres...,anticipation


In [ ]:
from sklearn.model_selection import GridSearchCV

print("1. TF-IDF Vektörizasyonu yapılıyor...")

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=5, max_df=0.7)
X = tfidf.fit_transform(df["Cleaned_Text"])
y = df["emotion"]

X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)
print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}\n")

print("2. SVM için hiperparametreler aranıyor")
svm_base = LinearSVC(class_weight='balanced', random_state=42, dual=False)

param_grid = {'C': [0.01, 0.05, 0.1, 0.5, 1, 5]}

grid_search = GridSearchCV(svm_base, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"--- Bulunan En İdeal Ayar: {grid_search.best_params_} ---\n")

print("3. Test seti üzerinde tahmin yapılıyor...")
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\n=== OPTİMİZE EDİLMİŞ SVM SONUÇLARI ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}\n")

print("Detaylı Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

test_results_df = df.loc[indices_test].copy()
test_results_df["Predicted_Emotion"] = y_pred
test_results_df["Used_Model"] = f"Optimized_SVM (C={grid_search.best_params_['C']})"

output_df = test_results_df[["Index", "movie_name", "Cleaned_Text", "emotion", "Predicted_Emotion", "Used_Model"]]
output_df.to_csv("../Results/Test_Predictions_3.csv", index=False)
print("\nTahminler '../Results/Test_Predictions_3.csv' olarak başarıyla kaydedildi!")

1. TF-IDF Vektörizasyonu yapılıyor...
Eğitim seti boyutu: (15452, 10000)
Test seti boyutu: (3864, 10000)

2. SVM için hiperparametreler aranıyor
--- Bulunan En İdeal Ayar: {'C': 0.5} ---

3. Test seti üzerinde tahmin yapılıyor...

=== OPTİMİZE EDİLMİŞ SVM SONUÇLARI ===
Accuracy : 0.5091
Precision: 0.5266
Recall   : 0.5091
F1-Score : 0.5147

Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

       anger       0.38      0.49      0.42       237
anticipation       0.48      0.47      0.48       530
     disgust       0.32      0.53      0.40       118
        fear       0.42      0.52      0.47       328
         joy       0.47      0.45      0.46       655
    optimism       0.35      0.39      0.37       397
     sadness       0.66      0.58      0.62      1597
    surprise       0.67      1.00      0.80         2

    accuracy                           0.51      3864
   macro avg       0.47      0.55      0.50      3864
weighted avg       0.53      0.